## Código 2 — Modelo NOCT y residuo

Este código toma la base generada por PVWatts para Santiago y calcula un modelo físico NOCT propio.

Se usa `dc` de PVWatts como potencia de referencia simulada (`p_ref_dc`) y se calcula `p_noct` usando `poa`, temperatura ambiente y parámetros estándar del panel.

Luego se calcula el residuo:

`epsilon = p_ref_dc - p_noct`

También se generan errores absolutos, errores relativos y una columna `is_valid_model` para filtrar las horas útiles en el diagnóstico y en el entrenamiento posterior.

In [2]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

INPUT_FILE = Path("data/01_pvwatts_santiago.csv")
OUT_FILE = Path("data/02_noct_vs_pvwatts_santiago.csv")
OUT_META = Path("data/02_noct_vs_pvwatts_santiago_metadata.json")

# Parámetros del sistema fotovoltaico
PDC0_W = 1000.0          # sistema FV de 1 kW DC
SYSTEM_LOSSES = 0.14     # debe coincidir con LOSSES = 14 del Código 1

# Parámetros del modelo NOCT
G_REF = 1000.0           # W/m2
T_REF = 25.0             # °C
NOCT = 45.0              # °C
GAMMA = -0.004           # 1/°C

# Filtros para diagnóstico/modelado
GHI_MIN = 10.0
POA_MIN = 10.0
P_REF_MIN_RELATIVE_ERROR = 50.0



# FUNCIONES
def validate_input(df: pd.DataFrame) -> None:
    required_columns = [
        "datetime_local",
        "year",
        "month",
        "day",
        "hour",
        "ghi",
        "dhi",
        "dni",
        "fd",
        "regimen_fd",
        "poa",
        "dc",
        "ac",
        "temperature",
        "tcell_pvwatts",
        "wind_speed",
        "albedo",
    ]

    missing = [col for col in required_columns if col not in df.columns]

    if missing:
        raise ValueError(f"Faltan columnas requeridas en la base PVWatts: {missing}")


def prepare_data(df: pd.DataFrame) -> pd.DataFrame:
    numeric_columns = [
        "year",
        "month",
        "day",
        "hour",
        "ghi",
        "dhi",
        "dni",
        "fd",
        "poa",
        "dc",
        "ac",
        "temperature",
        "tcell_pvwatts",
        "wind_speed",
        "albedo",
    ]

    for col in numeric_columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Orden fijo para gráficos y tablas posteriores
    df["regimen_fd"] = pd.Categorical(
        df["regimen_fd"],
        categories=["baja_fd", "mixto", "alta_fd"],
        ordered=True,
    )

    return df


def calculate_noct(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula el modelo NOCT base.

    Se calculan dos versiones:
    - p_noct_raw: NOCT sin pérdidas del sistema.
    - p_noct: NOCT con pérdidas del sistema, comparable con PVWatts dc.
    """

    df["tcell_noct"] = df["temperature"] + ((NOCT - 20.0) / 800.0) * df["poa"]

    temperature_factor = 1 + GAMMA * (df["tcell_noct"] - T_REF)

    df["p_noct_raw"] = (
        PDC0_W
        * (df["poa"] / G_REF)
        * temperature_factor
    )

    df["p_noct_raw"] = df["p_noct_raw"].fillna(0).clip(lower=0)

    # PVWatts fue descargado con losses=14,
    # por eso se aplica la misma pérdida al modelo NOCT.
    df["p_noct"] = df["p_noct_raw"] * (1 - SYSTEM_LOSSES)
    df["p_noct"] = df["p_noct"].fillna(0).clip(lower=0)

    return df


def calculate_errors(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula residuos respecto a PVWatts.

    p_ref_dc corresponde a la potencia DC simulada por PVWatts.
    No es una medición real de panel.
    """

    df["p_ref_dc"] = df["dc"]

    # Residuo principal usado en la tesis
    df["epsilon"] = df["p_ref_dc"] - df["p_noct"]
    df["abs_epsilon"] = df["epsilon"].abs()

    # Residuo auxiliar contra NOCT sin pérdidas
    df["epsilon_raw"] = df["p_ref_dc"] - df["p_noct_raw"]
    df["abs_epsilon_raw"] = df["epsilon_raw"].abs()

    # Error relativo: evitar dividir por potencias muy pequeñas
    df["relative_error"] = np.nan
    mask_relative = df["p_ref_dc"] > P_REF_MIN_RELATIVE_ERROR

    df.loc[mask_relative, "relative_error"] = (
        df.loc[mask_relative, "abs_epsilon"]
        / df.loc[mask_relative, "p_ref_dc"]
    )

    # Filas útiles para diagnóstico y ML
    df["is_valid_model"] = (
        (df["ghi"] > GHI_MIN)
        & (df["poa"] > POA_MIN)
        & (df["p_ref_dc"] > 0)
        & df["fd"].notna()
        & df["regimen_fd"].notna()
    )

    return df


def order_columns(df: pd.DataFrame) -> pd.DataFrame:
    columns = [
        "datetime_local",
        "year",
        "month",
        "day",
        "hour",

        "ghi",
        "dhi",
        "dni",
        "fd",
        "regimen_fd",

        "poa",
        "dc",
        "ac",
        "p_ref_dc",

        "p_noct_raw",
        "p_noct",

        "temperature",
        "tcell_pvwatts",
        "tcell_noct",
        "wind_speed",
        "albedo",

        "epsilon",
        "abs_epsilon",
        "epsilon_raw",
        "abs_epsilon_raw",
        "relative_error",
        "is_valid_model",
    ]

    return df[columns].copy()


def save_metadata() -> None:
    metadata = {
        "input_file": str(INPUT_FILE),
        "output_file": str(OUT_FILE),
        "description": "Comparación entre potencia DC simulada por PVWatts y modelo NOCT propio para Santiago.",
        "important_note": "p_ref_dc proviene de PVWatts y corresponde a potencia simulada, no a medición real.",
        "location": "Santiago, Chile",
        "noct_parameters": {
            "pdc0_w": PDC0_W,
            "g_ref": G_REF,
            "t_ref": T_REF,
            "noct": NOCT,
            "gamma": GAMMA,
            "system_losses_fraction": SYSTEM_LOSSES,
        },
        "valid_model_filter": {
            "ghi_min": GHI_MIN,
            "poa_min": POA_MIN,
            "p_ref_min_for_relative_error": P_REF_MIN_RELATIVE_ERROR,
        },
        "residual_definition": "epsilon = p_ref_dc - p_noct",
    }

    with open(OUT_META, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=4, ensure_ascii=False)


def print_diagnostics(df: pd.DataFrame) -> None:
    print("\nDimensiones:")
    print(df.shape)

    print("\nColumnas:")
    print(df.columns.tolist())

    print("\nFilas válidas para modelado:")
    print(df["is_valid_model"].value_counts(dropna=False))

    valid = df[df["is_valid_model"]].copy()

    print("\nConteo por regimen_fd:")
    print(valid["regimen_fd"].value_counts(dropna=False).sort_index())

    print("\nPromedios por regimen_fd:")
    summary = (
        valid
        .groupby("regimen_fd", observed=False)
        .agg(
            n=("p_ref_dc", "size"),
            fd_promedio=("fd", "mean"),
            p_ref_promedio_W=("p_ref_dc", "mean"),
            p_noct_promedio_W=("p_noct", "mean"),
            p_noct_raw_promedio_W=("p_noct_raw", "mean"),
            epsilon_promedio_W=("epsilon", "mean"),
            mae_W=("abs_epsilon", "mean"),
            error_relativo_promedio=("relative_error", "mean"),
        )
        .round(4)
    )

    print(summary)

    print("\nResumen global de variables clave:")
    print(
        valid[
            [
                "ghi",
                "dhi",
                "dni",
                "fd",
                "poa",
                "p_ref_dc",
                "p_noct",
                "p_noct_raw",
                "epsilon",
                "abs_epsilon",
                "relative_error",
                "temperature",
                "wind_speed",
            ]
        ].describe().round(4)
    )



# EJECUCION
def main() -> None:
    if not INPUT_FILE.exists():
        raise FileNotFoundError(
            f"No se encontró el archivo: {INPUT_FILE}. "
            "Primero ejecuta el Código 1 para Santiago."
        )

    print("Leyendo base PVWatts Santiago...")
    df = pd.read_csv(INPUT_FILE)

    print("Validando columnas...")
    validate_input(df)

    print("Preparando datos...")
    df = prepare_data(df)

    print("Calculando modelo NOCT con pérdidas del sistema...")
    df = calculate_noct(df)

    print("Calculando errores...")
    df = calculate_errors(df)

    df = order_columns(df)

    print("Guardando archivo...")
    OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(OUT_FILE, index=False)
    save_metadata()

    print("\nArchivo guardado:")
    print(OUT_FILE)

    print("\nMetadata guardada:")
    print(OUT_META)

    print_diagnostics(df)

    print("\nPrimeras 24 filas:")
    print(df.head(24))


if __name__ == "__main__":
    main()

Leyendo base PVWatts Santiago...
Validando columnas...
Preparando datos...
Calculando modelo NOCT con pérdidas del sistema...
Calculando errores...
Guardando archivo...



Archivo guardado:
data\02_noct_vs_pvwatts_santiago.csv

Metadata guardada:
data\02_noct_vs_pvwatts_santiago_metadata.json

Dimensiones:
(8760, 27)

Columnas:
['datetime_local', 'year', 'month', 'day', 'hour', 'ghi', 'dhi', 'dni', 'fd', 'regimen_fd', 'poa', 'dc', 'ac', 'p_ref_dc', 'p_noct_raw', 'p_noct', 'temperature', 'tcell_pvwatts', 'tcell_noct', 'wind_speed', 'albedo', 'epsilon', 'abs_epsilon', 'epsilon_raw', 'abs_epsilon_raw', 'relative_error', 'is_valid_model']

Filas válidas para modelado:
is_valid_model
False    4545
True     4215
Name: count, dtype: int64

Conteo por regimen_fd:
regimen_fd
baja_fd    1349
mixto      1126
alta_fd    1740
Name: count, dtype: int64

Promedios por regimen_fd:
               n  fd_promedio  p_ref_promedio_W  p_noct_promedio_W  \
regimen_fd                                                           
baja_fd     1349       0.1960          673.9032           659.3628   
mixto       1126       0.4873          361.5111           364.0207   
alta_fd     1

         datetime_local  year  month  day  hour          ghi    dhi    dni  \
0   2021-01-01 00:00:00  2021      1    1     0     0.000000    0.0    0.0   
1   2021-01-01 01:00:00  2021      1    1     1     0.000000    0.0    0.0   
2   2021-01-01 02:00:00  2021      1    1     2     0.000000    0.0    0.0   
3   2021-01-01 03:00:00  2021      1    1     3     0.000000    0.0    0.0   
4   2021-01-01 04:00:00  2021      1    1     4     0.000000    0.0    0.0   
5   2021-01-01 05:00:00  2021      1    1     5     3.000000    3.0    0.0   
6   2021-01-01 06:00:00  2021      1    1     6    63.590055   61.0   44.0   
7   2021-01-01 07:00:00  2021      1    1     7   235.015319  155.0  309.0   
8   2021-01-01 08:00:00  2021      1    1     8   448.615720  175.0  600.0   
9   2021-01-01 09:00:00  2021      1    1     9   670.323155  189.0  756.0   
10  2021-01-01 10:00:00  2021      1    1    10   859.849804  173.0  871.0   
11  2021-01-01 11:00:00  2021      1    1    11  1000.842673  13